# Component Modeling

This notebook starts the exploration of the component prediction model using the cleaned single-label data from the preprocessing stage. We want to establish a structured benchmark that shows improved predictive capabilities compared to a simple baseline and document what it gets right or wrong.

## 1. Setup and Ground Rules

We load the modeling libraries, define the split boundaries, and lock the structured feature lists.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    top_k_accuracy_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Robust pathing to directories and data
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'odi_component_single_label_cases.parquet'

# Model time and target definitions
TARGET_COL = 'component_group'
DATE_COL = 'ldate'
TRAIN_END = pd.Timestamp('2024-12-31')
VALID_END = pd.Timestamp('2025-12-31')
RANDOM_SEED = 42

# Constants
CAT_COLS = [
    'mfr_name',
    'maketxt',
    'modeltxt',
    'state',
    'cmpl_type',
    'drive_train',
    'fuel_sys',
    'fuel_type',
    'trans_type',
    'fire',
    'crash',
    'medical_attn',
    'vehicles_towed_yn'
]

NUM_COLS = [
    'yeartxt',
    'miles',
    'veh_speed',
    'injured',
    'lag_days_safe'
]

FLAG_COLS = [
    'miles_missing_flag',
    'veh_speed_missing_flag',
    'miles_zero_flag',
    'veh_speed_zero_flag'
]

FEATURE_COLS = CAT_COLS + NUM_COLS + FLAG_COLS

feature_inventory = pd.DataFrame(
    {
        'feature_block': ['categorical', 'numeric', 'flags', 'total'],
        'feature_count': [len(CAT_COLS), len(NUM_COLS), len(FLAG_COLS), len(FEATURE_COLS)]
    }
)

print("Structured benchmark feature inventory:")
display(feature_inventory)
DATA_PATH

Structured benchmark feature inventory:


,feature_block,feature_count
0,categorical,13
1,numeric,5
2,flags,4
3,total,22


WindowsPath('c:/Users/davis/OneDrive/Documents/VS Code Repos/NHTSA-ODI-Complaint-Analytics/data/processed/odi_component_single_label_cases.parquet')

## 2. Load And Audit The Modeling Table

We load the single-label case table created from `src/preprocessing/clean_complaints.py` and do an audit of the rows, dates, and target coverage to confirm setup. It also filters to the benchmark-ready subset flagged by `single_label_keep_flag` so the split matches the data we noted as ready for modeling.

In [ ]:
df = pd.read_parquet(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)

required_cols = FEATURE_COLS + [
    TARGET_COL,
    DATE_COL,
    'odino',
    'single_label_keep_flag',
    'component_group_fit_case_count'
]
missing_cols = [column for column in required_cols if column not in df.columns]
if missing_cols:
    missing_text = ', '.join(missing_cols)
    raise ValueError(f'Missing required columns: {missing_text}')

benchmark_ready_mask = df['single_label_keep_flag'].fillna(False).astype(bool)
filter_summary = pd.DataFrame([
    {
        'raw_rows': int(len(df)),
        'benchmark_ready_rows': int(benchmark_ready_mask.sum()),
        'rows_removed': int((~benchmark_ready_mask).sum()),
        'target_groups_after_filter': int(df.loc[benchmark_ready_mask, TARGET_COL].nunique()),
        'min_fit_case_count': int(df.loc[benchmark_ready_mask, 'component_group_fit_case_count'].min())
    }
])

df = df.loc[benchmark_ready_mask].copy().reset_index(drop=True)

for column in CAT_COLS:
    df[column] = df[column].astype(object)
    df[column] = df[column].where(pd.notna(df[column]), np.nan)

for column in NUM_COLS + FLAG_COLS:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Get general overview of data for this model
overview = pd.DataFrame([{
    'rows': int(len(df)),
    'columns': int(len(df.columns)),
    'target_groups': int(df[TARGET_COL].nunique()),
    'date_min': df[DATE_COL].min(),
    'date_max': df[DATE_COL].max()
}])

# Get the number of rows per year
year_counts = (
    df[DATE_COL]
    .dt.year
    .value_counts()
    .sort_index()
    .rename_axis('ldate_year')
    .reset_index(name='rows')
)

# Get the number of rows per component group
target_counts = (
    df[TARGET_COL]
    .value_counts()
    .rename_axis('component_group')
    .reset_index(name='rows')
)
target_counts['share'] = (target_counts['rows'] / len(df)).round(4)
target_counts['cum_share'] = target_counts['share'].cumsum().round(4)

feature_missingness = pd.DataFrame(
    {
        'feature': FEATURE_COLS,
        'missing_rate': [round(float(df[column].isna().mean()), 4) for column in FEATURE_COLS],
        'distinct_values': [int(df[column].nunique(dropna=True)) for column in FEATURE_COLS]
    }
).sort_values(['missing_rate', 'distinct_values'], ascending=[False, False]).reset_index(drop=True)

print("Benchmark-eligibility filter summary:")
display(filter_summary)

print("\nGeneral model data overview:")
display(overview)

print("\nRow counts per year in the single-label case table:")
display(year_counts)

print("\nLargest component groups and cumulative share:")
display(target_counts.head(15))

print("\nMissingness check across the structured benchmark features:")
display(feature_missingness.head(12))

Benchmark-eligibility filter summary:


,raw_rows,benchmark_ready_rows,rows_removed,target_groups_after_filter,min_fit_case_count
0,252161,251667,494,21,420



General model data overview:


,rows,columns,target_groups,date_min,date_max
0,251667,38,21,2020-01-01,2026-02-19



Row counts per year in the single-label case table:


,ldate_year,rows
0,2020,39352
1,2021,33326
2,2022,36240
3,2023,42157
4,2024,45584
5,2025,48013
6,2026,6995



Largest component groups and cumulative share:


,component_group,rows,share,cum_share
0,ENGINE / COOLING,52733,0.2095,0.2095
1,POWER TRAIN,31331,0.1245,0.3340
2,ELECTRICAL SYSTEM,27982,0.1112,0.4452
3,STEERING,21705,0.0862,0.5314
4,SERVICE BRAKES,20063,0.0797,0.6111
5,AIR BAGS,15776,0.0627,0.6738
6,FUEL / PROPULSION,15550,0.0618,0.7356
7,STRUCTURE,14580,0.0579,0.7935
8,VISIBILITY / WIPER,11724,0.0466,0.8401
9,EXTERIOR LIGHTING,7346,0.0292,0.8693



Missingness check across the structured benchmark features:


,feature,missing_rate,distinct_values
0,fuel_sys,0.9995,2
1,trans_type,0.9961,2
2,drive_train,0.9457,5
3,fuel_type,0.9454,5
4,miles,0.6430,16370
5,veh_speed,0.6110,175
6,lag_days_safe,0.0013,3079
7,yeartxt,0.0012,49
8,state,0.0001,62
9,modeltxt,0.0000,2372


## 3. Build The Time-Aware Split

We split by complaint receipt date so validation uses later complaints the model has not seen. This keeps the benchmark honest and leaves the partial `2026` slice untouched for future holdout testing.

In [ ]:
train_df = df.loc[df[DATE_COL] <= TRAIN_END].copy()
valid_df = df.loc[(df[DATE_COL] > TRAIN_END) & (df[DATE_COL] <= VALID_END)].copy()
holdout_df = df.loc[df[DATE_COL] > VALID_END].copy()

split_summary = pd.DataFrame([
    {
        'split': 'train',
        'rows': int(len(train_df)),
        'cases': int(train_df['odino'].nunique()),
        'date_min': train_df[DATE_COL].min(),
        'date_max': train_df[DATE_COL].max(),
        'target_groups': int(train_df[TARGET_COL].nunique())
    },
    {
        'split': 'valid_2025',
        'rows': int(len(valid_df)),
        'cases': int(valid_df['odino'].nunique()),
        'date_min': valid_df[DATE_COL].min(),
        'date_max': valid_df[DATE_COL].max(),
        'target_groups': int(valid_df[TARGET_COL].nunique())
    },
    {
        'split': 'holdout_2026',
        'rows': int(len(holdout_df)),
        'cases': int(holdout_df['odino'].nunique()),
        'date_min': holdout_df[DATE_COL].min(),
        'date_max': holdout_df[DATE_COL].max(),
        'target_groups': int(holdout_df[TARGET_COL].nunique())
    }
])

# Make sure no component group labels exist in the validation data that don't in the training data
unseen_valid_labels = sorted(set(valid_df[TARGET_COL]) - set(train_df[TARGET_COL]))

X_train = train_df[FEATURE_COLS].copy()
y_train = train_df[TARGET_COL].copy()
X_valid = valid_df[FEATURE_COLS].copy()
y_valid = valid_df[TARGET_COL].copy()

# Get the top component groups and what percent the make of the train and validation sets
train_top_groups = train_df[TARGET_COL].value_counts(normalize=True).head(12)
valid_top_groups = valid_df[TARGET_COL].value_counts(normalize=True)
target_shift = pd.DataFrame(
    {
        'component_group': train_top_groups.index,
        'train_share': np.round(train_top_groups.values, 4)
    }
)
target_shift['valid_share'] = target_shift['component_group'].map(valid_top_groups).fillna(0).round(4)
target_shift['share_delta'] = (target_shift['valid_share'] - target_shift['train_share']).round(4)

print("Summary of rows and targets per time split:")
display(split_summary)

print("Unseen validation labels:", unseen_valid_labels)

print("\nTop training groups and how their share shifts in validation:")
display(target_shift)

Summary of rows and targets per time split:


,split,rows,cases,date_min,date_max,target_groups
0,train,196659,196659,2020-01-01,2024-12-31,21
1,valid_2025,48013,48013,2025-01-01,2025-12-31,21
2,holdout_2026,6995,6995,2026-01-01,2026-02-19,20


Unseen validation labels: []

Top training groups and how their share shifts in validation:


,component_group,train_share,valid_share,share_delta
0,ENGINE / COOLING,0.1965,0.2600,0.0635
1,POWER TRAIN,0.1168,0.1498,0.0330
2,ELECTRICAL SYSTEM,0.1075,0.1201,0.0126
3,STEERING,0.0918,0.0679,-0.0239
4,SERVICE BRAKES,0.0830,0.0687,-0.0143
5,AIR BAGS,0.0687,0.0420,-0.0267
6,FUEL / PROPULSION,0.0633,0.0563,-0.0070
7,STRUCTURE,0.0579,0.0600,0.0021
8,VISIBILITY / WIPER,0.0505,0.0317,-0.0188
9,EXTERIOR LIGHTING,0.0314,0.0203,-0.0111


## 4. Naive Baseline

Start with the simplest possible reference point: always predict the most common training label. This gives the stronger models a concrete floor to beat.

In [4]:
train_target_counts = y_train.value_counts()
majority_label = train_target_counts.idxmax()
top3_labels = train_target_counts.head(3).index.tolist()

naive_pred = pd.Series(majority_label, index=y_valid.index)
naive_summary = pd.DataFrame([{
    'model': 'Most Frequent Label',
    'top_1_accuracy': round(accuracy_score(y_valid, naive_pred), 4),
    'macro_f1': round(f1_score(y_valid, naive_pred, average='macro'), 4),
    'top_3_accuracy': round(y_valid.isin(top3_labels).mean(), 4)
}])

print("Most frequent training label:", majority_label)
print("Top 3 training labels:", top3_labels)
print("\nNaive baseline model evaluation:")
display(naive_summary)

Most frequent training label: ENGINE / COOLING
Top 3 training labels: ['ENGINE / COOLING', 'POWER TRAIN', 'ELECTRICAL SYSTEM']

Naive baseline model evaluation:


,model,top_1_accuracy,macro_f1,top_3_accuracy
0,Most Frequent Label,0.26,0.0197,0.53


## 5. Logistic Regression Baseline

Start with a simple linear model on the current core structured features. This gives a transparent baseline before we move to the stronger tree model.



In [ ]:
# Imputation and encoding pipeline for categorical features
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Imputation and scaling pipeline for numeric features
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler(with_mean=False))
])

# Column transformer to divert columns to correct pipeline
prep = ColumnTransformer([
    ('cat', cat_pipe, CAT_COLS),
    ('num', num_pipe, NUM_COLS + FLAG_COLS)
])

# Logistic regression pipeline to prevent leakage
logit_model = Pipeline([
    ('prep', prep),
    ('model', LogisticRegression(
        solver='saga',
        class_weight='balanced',
        max_iter=300,
        tol=5e-3,
        random_state=RANDOM_SEED
    ))
])

start = perf_counter()
logit_model.fit(X_train, y_train)
fit_seconds = perf_counter() - start
fit_seconds

42.07710870000301

### 5.1 Logistic Overall Metrics

After fitting, we score the logistic model on both train and validation.

In [6]:
def score_split(split_name, y_true, pred, proba, classes):
    return {
        'split': split_name,
        'rows': int(len(y_true)),
        'top_1_accuracy': round(accuracy_score(y_true, pred), 4),
        'macro_f1': round(f1_score(y_true, pred, average='macro'), 4),
        'top_3_accuracy': round(top_k_accuracy_score(y_true, proba, labels=classes, k=3), 4)
    }

train_pred = logit_model.predict(X_train)
train_proba = logit_model.predict_proba(X_train)
valid_pred = logit_model.predict(X_valid)
valid_proba = logit_model.predict_proba(X_valid)
classes = logit_model.named_steps['model'].classes_

baseline_results = pd.DataFrame([
    score_split('train', y_train, train_pred, train_proba, classes),
    score_split('valid_2025', y_valid, valid_pred, valid_proba, classes)
])

print(f"Fit time (seconds): {fit_seconds:.2f}")
print("\nLogistic Regression baseline results:")
display(baseline_results)

Fit time (seconds): 42.08

Logistic Regression baseline results:


,split,rows,top_1_accuracy,macro_f1,top_3_accuracy
0,train,196659,0.1922,0.1634,0.4252
1,valid_2025,48013,0.1453,0.1141,0.3657


### 5.2 Logistic Class Level Review

Overall metrics can hide weak classes, so we break the logistic baseline down by component group. That gives us an early view of which labels are easy or hard with a linear model.

In [7]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_valid,
    valid_pred,
    labels=classes,
    zero_division=0
)

class_results = pd.DataFrame(
    {
        'component_group': classes,
        'support': support,
        'precision': np.round(precision, 4),
        'recall': np.round(recall, 4),
        'f1': np.round(f1, 4)
    }
).sort_values(['support', 'f1'], ascending=[False, False]).reset_index(drop=True)

print("Logistic Regression baseline evaluation by class:")
display(class_results)

Logistic Regression baseline evaluation by class:


,component_group,support,precision,recall,f1
0,ENGINE / COOLING,12485,0.6200,0.1241,0.2069
1,POWER TRAIN,7193,0.6315,0.1492,0.2413
2,ELECTRICAL SYSTEM,5767,0.3219,0.0277,0.0511
3,SERVICE BRAKES,3298,0.3438,0.2468,0.2873
4,STEERING,3258,0.2261,0.1529,0.1824
5,STRUCTURE,2882,0.1131,0.0725,0.0884
6,FUEL / PROPULSION,2702,0.1112,0.1029,0.1069
7,AIR BAGS,2016,0.1476,0.3100,0.2000
8,VISIBILITY / WIPER,1520,0.1769,0.0757,0.1060
9,BACK OVER PREVENTION,1304,0.1091,0.3765,0.1692


### 5.3 Logistic Confusion Review

This confusion matrix view focuses on the largest classes in the training data. It makes the biggest routing mistakes easier to see than the overall metrics alone.

In [ ]:
focus_groups = train_df[TARGET_COL].value_counts().head(12).index.tolist()
valid_focus = y_valid.where(y_valid.isin(focus_groups))
valid_pred_focus = pd.Series(valid_pred, index=y_valid.index).where(lambda s: s.isin(focus_groups))

confusion_counts = pd.crosstab(
    pd.Categorical(valid_focus, categories=focus_groups),
    pd.Categorical(valid_pred_focus, categories=focus_groups),
    dropna=False
)
confusion_share = confusion_counts.div(
    confusion_counts.sum(axis=1).replace(0, np.nan),
    axis=0
).round(3)

print("Focused confusion counts for the 12 largest training groups")
display(confusion_counts)

print("Focused row-normalized confusion shares")
display(confusion_share)

Focused confusion counts for the 12 largest training groups


col_0,ENGINE / COOLING,POWER TRAIN,ELECTRICAL SYSTEM,STEERING,SERVICE BRAKES,AIR BAGS,FUEL / PROPULSION,STRUCTURE,VISIBILITY / WIPER,EXTERIOR LIGHTING,SUSPENSION,FORWARD COLLISION AVOIDANCE
row_0,,,,,,,,,,,,
NaN,73,39,52,49,128,269,123,181,62,294,19,184
ENGINE / COOLING,1550,288,98,534,406,1089,431,268,68,1365,48,515
POWER TRAIN,187,1073,43,341,400,595,160,347,62,439,25,248
ELECTRICAL SYSTEM,184,62,160,108,133,549,581,186,123,589,18,237
STEERING,82,57,18,498,187,212,342,49,13,114,23,510
SERVICE BRAKES,75,61,16,124,814,254,171,217,34,176,15,103
AIR BAGS,16,8,10,15,11,625,76,135,65,179,10,80
FUEL / PROPULSION,133,44,39,106,93,208,278,110,15,177,38,217
STRUCTURE,75,11,15,301,39,117,116,209,25,276,17,83


Focused row-normalized confusion shares


col_0,ENGINE / COOLING,POWER TRAIN,ELECTRICAL SYSTEM,STEERING,SERVICE BRAKES,AIR BAGS,FUEL / PROPULSION,STRUCTURE,VISIBILITY / WIPER,EXTERIOR LIGHTING,SUSPENSION,FORWARD COLLISION AVOIDANCE
row_0,,,,,,,,,,,,
NaN,0.050,0.026,0.035,0.033,0.087,0.183,0.084,0.123,0.042,0.200,0.013,0.125
ENGINE / COOLING,0.233,0.043,0.015,0.080,0.061,0.164,0.065,0.040,0.010,0.205,0.007,0.077
POWER TRAIN,0.048,0.274,0.011,0.087,0.102,0.152,0.041,0.089,0.016,0.112,0.006,0.063
ELECTRICAL SYSTEM,0.063,0.021,0.055,0.037,0.045,0.187,0.198,0.063,0.042,0.201,0.006,0.081
STEERING,0.039,0.027,0.009,0.237,0.089,0.101,0.162,0.023,0.006,0.054,0.011,0.242
SERVICE BRAKES,0.036,0.030,0.008,0.060,0.395,0.123,0.083,0.105,0.017,0.085,0.007,0.050
AIR BAGS,0.013,0.007,0.008,0.012,0.009,0.508,0.062,0.110,0.053,0.146,0.008,0.065
FUEL / PROPULSION,0.091,0.030,0.027,0.073,0.064,0.143,0.191,0.075,0.010,0.121,0.026,0.149
STRUCTURE,0.058,0.009,0.012,0.234,0.030,0.091,0.090,0.163,0.019,0.215,0.013,0.065


## 6. CatBoost Benchmark

CatBoost is the selected structured benchmark due to how it handles mixed categoricals, missingness, and nonlinear interactions without needing extra feature engineering. This section uses the same time-aware split and feature set so the comparison against the earlier baselines stays fair.

In [ ]:
def prep_catboost_frame(frame):
    frame = frame.copy()
    for column in CAT_COLS:
        frame[column] = frame[column].astype('string').fillna('__MISSING__')
    for column in NUM_COLS + FLAG_COLS:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')
    return frame


X_train_cat = prep_catboost_frame(X_train)
X_valid_cat = prep_catboost_frame(X_valid)

cat_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='MultiClass',
    auto_class_weights='Balanced',
    bootstrap_type='Bernoulli',
    iterations=250,
    learning_rate=0.08,
    depth=8,
    l2_leaf_reg=6.0,
    subsample=0.8,
    random_seed=RANDOM_SEED,
    allow_writing_files=False,
    verbose=25,
    task_type='GPU',
    devices='0'
)

start = perf_counter()
cat_model.fit(
    X_train_cat,
    y_train,
    cat_features=CAT_COLS,
    eval_set=(X_valid_cat, y_valid),
    use_best_model=True,
    early_stopping_rounds=30
)
cat_fit_seconds = perf_counter() - start
cat_fit_seconds

0:	learn: 2.9557220	test: 2.9773219	best: 2.9773219 (0)	total: 64.6ms	remaining: 16.1s


25:	learn: 2.3407323	test: 2.5414498	best: 2.5414498 (25)	total: 1.48s	remaining: 12.7s


50:	learn: 2.1827197	test: 2.4263612	best: 2.4263612 (50)	total: 2.81s	remaining: 11s


75:	learn: 2.0736561	test: 2.3691587	best: 2.3691587 (75)	total: 4.11s	remaining: 9.41s


100:	learn: 2.0057674	test: 2.3370371	best: 2.3370371 (100)	total: 5.45s	remaining: 8.04s


...

175:	learn: 1.8648106	test: 2.3027131	best: 2.3027131 (175)	total: 9.37s	remaining: 3.94s


200:	learn: 1.8390373	test: 2.2964905	best: 2.2964905 (200)	total: 10.6s	remaining: 2.58s


225:	learn: 1.8151581	test: 2.2932554	best: 2.2931424 (220)	total: 11.8s	remaining: 1.25s


249:	learn: 1.7871652	test: 2.2890125	best: 2.2888478 (245)	total: 13s	remaining: 0us
bestTest = 2.288847762
bestIteration = 245
Shrink model to first 246 iterations.


15.302769700007048

### 6.1 CatBoost Overall Metrics

We score the CatBoost benchmark on both train and validation then compare it against the earlier baselines. We want to know if the stronger structured model gives a meaningful predictive increase, not just a tiny incremental gain.

In [ ]:
cat_train_pred = pd.Series(cat_model.predict(X_train_cat).ravel(), index=y_train.index)
cat_train_proba = cat_model.predict_proba(X_train_cat)
cat_valid_pred = pd.Series(cat_model.predict(X_valid_cat).ravel(), index=y_valid.index)
cat_valid_proba = cat_model.predict_proba(X_valid_cat)
cat_classes = cat_model.classes_

catboost_results = pd.DataFrame([
    score_split('train', y_train, cat_train_pred, cat_train_proba, cat_classes),
    score_split('valid_2025', y_valid, cat_valid_pred, cat_valid_proba, cat_classes)
])

naive_compare = pd.DataFrame([
    {
        'model': 'Most Frequent Label',
        'split': 'valid_2025',
        'rows': int(len(y_valid)),
        'top_1_accuracy': round(accuracy_score(y_valid, naive_pred), 4),
        'macro_f1': round(f1_score(y_valid, naive_pred, average='macro'), 4),
        'top_3_accuracy': round(y_valid.isin(top3_labels).mean(), 4)
    }
])

logit_compare = baseline_results.copy()
logit_compare.insert(0, 'model', 'Logistic Regression')
catboost_compare = catboost_results.copy()
catboost_compare.insert(0, 'model', 'CatBoost')
model_compare = pd.concat([naive_compare, logit_compare, catboost_compare], ignore_index=True)

valid_compare = model_compare.loc[model_compare['split'].eq('valid_2025')].copy().reset_index(drop=True)
naive_macro_f1 = float(valid_compare.loc[valid_compare['model'].eq('Most Frequent Label'), 'macro_f1'].iloc[0])
valid_compare['macro_f1_gain_vs_naive'] = (valid_compare['macro_f1'] - naive_macro_f1).round(4)

print(f"CatBoost fit time (seconds): {cat_fit_seconds:.2f}")
print("\nCatBoost results:")
display(catboost_results)

print("\nValidation comparison across the structured benchmark path:")
display(valid_compare)

CatBoost fit time (seconds): 15.30

CatBoost results:


,split,rows,top_1_accuracy,macro_f1,top_3_accuracy
0,train,196659,0.3868,0.3286,0.6273
1,valid_2025,48013,0.2818,0.2174,0.4972



Validation comparison across the structured benchmark path:


,model,split,rows,top_1_accuracy,macro_f1,top_3_accuracy,macro_f1_gain_vs_naive
0,Most Frequent Label,valid_2025,48013,0.2600,0.0197,0.5300,0.0000
1,Logistic Regression,valid_2025,48013,0.1453,0.1141,0.3657,0.0944
2,CatBoost,valid_2025,48013,0.2818,0.2174,0.4972,0.1977


### 6.2 CatBoost Class-Level Review

Break the CatBoost results down by component group to see where the benchmark is strong and where it is still weak. If we later add text work, it should target the classes that structured features still struggle to separate.

In [11]:
cat_precision, cat_recall, cat_f1, cat_support = precision_recall_fscore_support(
    y_valid,
    cat_valid_pred,
    labels=cat_classes,
    zero_division=0
)

cat_class_results = pd.DataFrame(
    {
        'component_group': cat_classes,
        'support': cat_support,
        'precision': np.round(cat_precision, 4),
        'recall': np.round(cat_recall, 4),
        'f1': np.round(cat_f1, 4)
    }
).sort_values(['support', 'f1'], ascending=[False, False]).reset_index(drop=True)

print("CatBoost validation evaluation by class:")
display(cat_class_results)

CatBoost validation evaluation by class:


,component_group,support,precision,recall,f1
0,ENGINE / COOLING,12485,0.7733,0.2536,0.3819
1,POWER TRAIN,7193,0.5333,0.3042,0.3874
2,ELECTRICAL SYSTEM,5767,0.3677,0.1602,0.2232
3,SERVICE BRAKES,3298,0.5199,0.2978,0.3786
4,STEERING,3258,0.5172,0.3874,0.4430
5,STRUCTURE,2882,0.4975,0.3470,0.4088
6,FUEL / PROPULSION,2702,0.2103,0.1806,0.1943
7,AIR BAGS,2016,0.4686,0.4038,0.4338
8,VISIBILITY / WIPER,1520,0.2240,0.2461,0.2345
9,BACK OVER PREVENTION,1304,0.1600,0.6342,0.2556


### 6.3 CatBoost Confusion And Importance Review

Final diagnostic block to show the largest class confusions and the feature-importance profile. Together they show what the structured benchmark is learning and where the remaining ambiguity likely lives.

In [ ]:
cat_valid_focus = y_valid.where(y_valid.isin(focus_groups))
cat_pred_focus = pd.Series(cat_valid_pred, index=y_valid.index).where(lambda s: s.isin(focus_groups))

cat_confusion_counts = pd.crosstab(
    pd.Categorical(cat_valid_focus, categories=focus_groups),
    pd.Categorical(cat_pred_focus, categories=focus_groups),
    dropna=False
)
cat_confusion_share = cat_confusion_counts.div(
    cat_confusion_counts.sum(axis=1).replace(0, np.nan),
    axis=0
).round(3)

cat_importance = pd.DataFrame(
    {
        'feature': FEATURE_COLS,
        'importance': np.round(cat_model.get_feature_importance(), 4)
    }
).sort_values('importance', ascending=False).reset_index(drop=True)

print("CatBoost focused confusion counts for the 12 largest training groups")
display(cat_confusion_counts)

print("Focused row-normalized confusion shares")
display(cat_confusion_share)

print("CatBoost feature importance")
display(cat_importance.head(20))

CatBoost focused confusion counts for the 12 largest training groups


col_0,ENGINE / COOLING,POWER TRAIN,ELECTRICAL SYSTEM,STEERING,SERVICE BRAKES,AIR BAGS,FUEL / PROPULSION,STRUCTURE,VISIBILITY / WIPER,EXTERIOR LIGHTING,SUSPENSION,FORWARD COLLISION AVOIDANCE
row_0,,,,,,,,,,,,
NaN,74,71,139,69,78,130,94,84,157,84,51,172
ENGINE / COOLING,3166,971,536,186,204,159,474,367,216,834,102,727
POWER TRAIN,319,2188,229,162,241,80,288,151,184,246,96,278
ELECTRICAL SYSTEM,128,268,924,158,115,155,315,147,233,289,101,379
STEERING,81,160,118,1262,84,76,151,51,52,71,83,336
SERVICE BRAKES,82,153,91,194,982,93,154,56,66,104,67,156
AIR BAGS,26,31,41,30,32,814,174,26,42,98,35,81
FUEL / PROPULSION,91,122,161,118,53,68,488,48,82,78,105,249
STRUCTURE,56,30,93,122,32,67,71,1000,102,104,73,86


Focused row-normalized confusion shares


col_0,ENGINE / COOLING,POWER TRAIN,ELECTRICAL SYSTEM,STEERING,SERVICE BRAKES,AIR BAGS,FUEL / PROPULSION,STRUCTURE,VISIBILITY / WIPER,EXTERIOR LIGHTING,SUSPENSION,FORWARD COLLISION AVOIDANCE
row_0,,,,,,,,,,,,
NaN,0.062,0.059,0.116,0.057,0.065,0.108,0.078,0.070,0.131,0.070,0.042,0.143
ENGINE / COOLING,0.399,0.122,0.067,0.023,0.026,0.020,0.060,0.046,0.027,0.105,0.013,0.092
POWER TRAIN,0.071,0.490,0.051,0.036,0.054,0.018,0.065,0.034,0.041,0.055,0.022,0.062
ELECTRICAL SYSTEM,0.040,0.083,0.288,0.049,0.036,0.048,0.098,0.046,0.073,0.090,0.031,0.118
STEERING,0.032,0.063,0.047,0.500,0.033,0.030,0.060,0.020,0.021,0.028,0.033,0.133
SERVICE BRAKES,0.037,0.070,0.041,0.088,0.447,0.042,0.070,0.025,0.030,0.047,0.030,0.071
AIR BAGS,0.018,0.022,0.029,0.021,0.022,0.569,0.122,0.018,0.029,0.069,0.024,0.057
FUEL / PROPULSION,0.055,0.073,0.097,0.071,0.032,0.041,0.293,0.029,0.049,0.047,0.063,0.150
STRUCTURE,0.031,0.016,0.051,0.066,0.017,0.036,0.039,0.545,0.056,0.057,0.040,0.047


CatBoost feature importance


,feature,importance
0,mfr_name,21.3479
1,modeltxt,18.8970
2,yeartxt,16.1463
3,maketxt,11.2778
4,veh_speed,7.6494
5,cmpl_type,6.0551
6,lag_days_safe,4.3128
7,miles,4.0794
8,state,3.4385
9,miles_missing_flag,1.6740


## 7. Next Steps

This is as far as this notebook alone goes, we discovered enough to get a time-aware structured benchmark on the single-label case table, plus enough diagnostics to see where structured-only routing still struggles. The next steps required using a cloud environment connected with better runtime hardware accelerators due to them being computationally expensive.

The follow-on work now lives in `notebooks/archive/`:

- `component_single_structured_baseline.py` hardens this same structured benchmark into saved benchmark artifacts
- `component_feature_wave1.py` extends the benchmark with the first structured feature-wave screen and prune pass
- `component_text_wave2.py` adds the text sidecar and compares text-only, text plus structured, and late-fusion families

The outcome from these was that the best model included a slightly expanded feature set with a later text fusion. This model was then tuned and the optimal parameters were found. The official single-label and multi-label pipelines only settle on the final model after those later extensions and now live in `src\modeling\component_single_text_calibrated.py` and `src\modeling\component_multi_routing.py`.